In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os, glob, cv2
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
from read_images import read_images
from pre_processing import process_image_set
from pyfing import LeaderParameters, Leader, Snfoe
from minutiae_detection import detect_annotated_minutiae, detect_minutiae_automatic
from matplotlib.lines import Line2D
from matplotlib.ticker import MultipleLocator
from matplotlib.collections import LineCollection
from pyfing.utils.drawing import draw_orientations, draw_minutiae
from matplotlib.colors import LinearSegmentedColormap

In [ ]:
# Distal axis angles in radian
"""
Clay Clear (CamClose/LightClose/LowAngle) ->  1.629
Clay Faint (CamClose/LightFar/HighAngle)  ->  1.585
BK-NM-12006-19 - Clear                    ->  2.213
BK-NM-12006-19 - Faint                    ->  1.886
BK-AM-51-1-E                              ->  2.180
BK-AM-51-1-G                              ->  1.701
"""

In [ ]:
# detect minutiae
def detect_minutiae(image_folder, max_block, minutia_quality_threshold):
    acquisitions = read_images(image_folder)

    for i, imgset in enumerate(acquisitions):
        imgset = process_image_set(imgset, histEq=True, target_ridge_distance=10,
                                   max_block=max_block)
        
    params = LeaderParameters(minutia_quality_threshold=minutia_quality_threshold)
    ldr = Leader(parameters=params)
    snfoe = Snfoe()

    for j, acquisition in enumerate(acquisitions):
        # Keep only the four cardinal illumination directions
        acquisition = [img for img in acquisition
                       if img.direction.lower() in ('north', 'south', 'east', 'west')]

        # Detect annotations and automatic minutiae once per image
        for img in acquisition:
            if getattr(img, 'annotations', None) is None:
                theta, _ = snfoe.run(img.normalisedROI, img.normalisedROIMask)
                img.theta = theta
                img.annotations, _, _ = detect_annotated_minutiae(img, theta, min_area=1, 
                                                                  blur_ksize=False)

            if getattr(img, 'minutiae', None) is None:
                img.minutiae = detect_minutiae_automatic(img, ldr)

    return acquisitions

# minutiae detection spatial maps
def plot_minutiae_map(acquisition, source='annotations',
                      marker_size=35, saveto=None):

    styles = {'north': ('#00FFFF', 'o'),  # Bright blue — circle
              'east':  ('#39FF14', 's'),  # Bright orange1849, 1786 — square
              'south': ('#FF1493', '*'),  # Bright green — star
              'west':  ('#0066FF', 'D')   # Bright magenta — diamond
             }

    # Use the first image as the reference background
    reference = acquisition[1]
    background = reference.normalisedROI

    height, width = background.shape[:2]

    # Reference ROI origin in aligned, scaled coordinates
    y0, x0 = np.asarray(reference.roiCoords) * reference.median_scale

    fig, ax = plt.subplots(figsize=(7, 7))

    ax.imshow(background, cmap='gray', origin='upper')

    legend_handles = []

    for direction, (color, marker) in styles.items():
        img = next((img for img in acquisition if img.direction.lower() == direction), None)

        if img is None:
            continue

        minutiae = getattr(img, source)

        if not minutiae:
            continue

        # Convert aligned coordinates to reference ROI coordinates
        x = np.array([m.x - x0 for m in minutiae])
        y = np.array([m.y - y0 for m in minutiae])

        # Only display minutiae within the reference image
        valid = ((x >= 0) & (x < width) & (y >= 0) & (y < height))

        ax.scatter(x[valid], y[valid], s=marker_size * 2 if direction == 'south' else marker_size,
                   marker=marker, c=color, edgecolors='white', linewidths=0.6, alpha=0.9, zorder=2)

        legend_handles.append(Line2D([0], [0], marker=marker, linestyle='None', markerfacecolor=color,
                                     markeredgecolor='white', markersize=12, label=direction.capitalize()))

    ax.set_xlim(-0.5, width - 0.5)
    ax.set_ylim(height - 0.5, -0.5)
    ax.set_aspect('equal')
    ax.axis('off')

    ax.legend(handles=legend_handles, loc='upper center', bbox_to_anchor=(0.5, -0.02), ncol=4, frameon=False)

    plt.tight_layout()

    if saveto is not None:plt.savefig(saveto, dpi=300, bbox_inches='tight')

    plt.close(fig)

In [ ]:
# overall detection table1849, 1786
def overall_detection_table(results):
    single_view = results[results['NumCombi'] == 1]

    columns = {'Annotated': 'NumReference', 'Detected': 'NumReferenceDetected',
               'Correct': 'CorrectDetections', 'Missed': 'MissedMinutiae',
               'Spurious': 'SpuriousDetections'}

    summary = pd.DataFrame([{name: f"{single_view[col].mean():.1f} ± {single_view[col].std():.1f}"
                             for name, col in columns.items()}])

    return summary

In [ ]:
# detected minutiae by illumination direction
def direction_detection_table(results):
    single_view = results[results['NumCombi'] == 1].copy()

    # Standardize direction names
    single_view['Direction'] = (single_view['Direction'].str.strip().str.capitalize())

    summary = (single_view.groupby('Direction').agg(AnnotatedMean=('NumReference', 'mean'), AnnotatedStd=('NumReference', 'std'),
                                                    DetectedMean=('NumReferenceDetected', 'mean'), DetectedStd=('NumReferenceDetected', 'std'))
        .reindex(['North', 'East', 'South', 'West']))

    summary[['AnnotatedStd', 'DetectedStd']] = (summary[['AnnotatedStd', 'DetectedStd']].fillna(0))

    table = pd.DataFrame({'Direction': summary.index,
                          'Annotated': [f'{mean:.1f} ± {std:.1f}'
                                        for mean, std in zip(summary['AnnotatedMean'], summary['AnnotatedStd'])],
                          'Detected': [f'{mean:.1f} ± {std:.1f}'
                                       for mean, std in zip(summary['DetectedMean'], summary['DetectedStd'])]})

    return table

In [ ]:
def plot_fusion_set_size(df, source='annotations', saveto=None):
    value_col = {'annotations': 'AnnotationFinalSetSize','minutiae': 'DetectionFinalSetSize'}[source]

    for col in ['NumCombi', 'AngularSeparation', value_col]:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    fig, ax = plt.subplots(figsize=(7, 5))

    # One mean and min–max error bar per number of views
    summary = (df.groupby('NumCombi')[value_col].agg(['mean', 'min', 'max'])
        .reindex([1, 2, 3, 4]).dropna())

    x = summary.index.to_numpy(dtype=float)
    mean = summary['mean'].to_numpy()

    ax.errorbar(x, mean, yerr=[mean - summary['min'].to_numpy(), summary['max'].to_numpy() - mean],
                fmt='o', color='black', markersize=7, capsize=5, linewidth=1.5, zorder=4)

    # Individual two-view observations by angular separation
    pairs = df[df['NumCombi'] == 2]

    for separation, color, marker, offset in [(90, '#00BFFF', 'o', -0.06), (180, '#FF3030', '^', 0.06)]:
        subset = pairs[pairs['AngularSeparation'] == separation]

        ax.scatter(np.full(len(subset), 2.0 + offset), subset[value_col], s=65, marker=marker,
                   facecolors=color if separation == 90 else 'none', edgecolors='black' if separation == 90 else color,
                   linewidths=1.5, alpha=0.9, zorder=5)

    # Axes
    ax.set_xlabel('Number of illumiantion directions')
    ax.set_ylabel('Final minutiae set size')

    ax.set_xticks([1, 2, 3, 4])
    ax.set_xlim(0.7, 4.3)

    ymax = df[value_col].max()
    ax.set_ylim(0, ymax * 1.1 if ymax > 0 else 1)

    ax.yaxis.set_major_locator(MultipleLocator(5))

    # Light grid behind the data
    ax.set_axisbelow(True)
    ax.grid(which='major', axis='both', color='lightgray', linestyle='--', linewidth=0.6, alpha=0.5)

    # Legend
    legend = [Line2D([0], [0], marker='o', linestyle='None', color='black', markersize=7, label='Mean (min–max)'),
              Line2D([0], [0], marker='o', linestyle='None', markerfacecolor='#00BFFF', markeredgecolor='black',
                     markersize=8, label='90° pair'),
              Line2D([0], [0], marker='^', linestyle='None', markerfacecolor='none', markeredgecolor='#FF3030', 
                     markeredgewidth=1.5, markersize=9, label='180° pair')]

    ax.legend(handles=legend, frameon=False)

    plt.tight_layout()

    if saveto is not None:plt.savefig(saveto, dpi=300, bbox_inches='tight')

    # plt.show()
    plt.close(fig)

In [ ]:
def plot_minutiae_by_illumination(acquisitions, distal_angle, saveto=None):
    """
    acquisitions: list of acquisitions, where each acquisition
                  contains four ImageSet objects (N, E, S, W).
    distal_angle: shared distal-axis angle in radians.
    """
    directions = ['north', 'east', 'south', 'west']

    illumination_angles = {'north': distal_angle, 'east': distal_angle + np.pi / 2,
                           'south': distal_angle + np.pi, 'west': distal_angle - np.pi / 2,}

    counts_by_direction = {d: [] for d in directions}
    angles_by_direction = {d: [] for d in directions}

    # Calculate counts and angles for each acquisition
    for imagesets in acquisitions:
        images_by_direction = {img.direction: img for img in imagesets}

        for direction in directions:
            img = images_by_direction[direction]

            # Number of detected minutiae
            counts_by_direction[direction].append(len(img.minutiae))

            # Local ridge orientations
            theta = np.asarray(img.theta, dtype=float)
            theta = theta[np.isfinite(theta)]

            # Unsigned angle between ridge flow and illumination (0–90°)
            differences = (theta - illumination_angles[direction] + np.pi / 2) % np.pi - np.pi / 2

            local_angles = np.degrees(np.abs(differences))

            angles_by_direction[direction].append(np.median(local_angles) if local_angles.size else np.nan)

    # Mean values across acquisitions
    counts = [np.mean(counts_by_direction[d]) for d in directions]

    angles = [np.mean([a for a in angles_by_direction[d] if np.isfinite(a)]) if any(np.isfinite(a) for a in angles_by_direction[d])
              else np.nan for d in directions]

    # Plot
    fig, ax = plt.subplots(figsize=(5, 3))
    y = np.arange(len(directions))

    colors = {'north': '#4C78A8', 'east': '#F58518', 'south': '#54A24B', 'west': '#B279A2',}

    bars = ax.barh(y, counts, height=0.62, color=[colors[d] for d in directions], zorder=2)

    # Direction labels at the ends of the bars
    xmax = max(counts)
    offset = max(xmax * 0.025, 0.5)

    for bar, direction in zip(bars, directions):
        ax.text(bar.get_width() + offset, bar.get_y() + bar.get_height() / 2,
                direction.capitalize(), ha='left', va='center', fontsize=9)

    # Mean ridge–illumination angles on the y-axis
    ax.set_yticks(y)
    ax.set_yticklabels([f'{angle:.0f}°' if np.isfinite(angle) else 'N/A' for angle in angles])
    ax.invert_yaxis()

    ax.set_xlabel('Mean number of detected minutiae')
    ax.set_ylabel('Ridge angle relative\nto illumination (°)')

    # Extra space to the right of the bars and labels
    upper = int(np.ceil((xmax + offset + 12) / 2) * 2)
    ax.set_xlim(0, upper)

    # X-axis ticks and vertical gridlines
    ax.set_xticks(np.arange(0, upper + 1, 5))
    ax.grid(axis='x', color='0.85', linewidth=0.7, zorder=0)
    ax.tick_params(axis='y', length=0)
    ax.spines[['top', 'right', 'left']].set_visible(False)

    fig.tight_layout()

    if saveto:
        fig.savefig(saveto, dpi=300, bbox_inches='tight')

    plt.show()

In [ ]:
result_folder = '../Figures/DirectionEval/BK-NM-12006-19/Clear/CSVS/'
image_folder = '../Data/BK-NM-12006-19/Clear/Aligned/'
prefix = 'bk-nm-12006-19'
result_file = f'{prefix}_minutiae_gain_evaluation.csv'
save_folder = './'#../Figures/DirectionEval/BK-NM-12006-19/Clear/Figures/'

In [ ]:
# read file
df = pd.read_csv(f'{result_folder}/{result_file}')

In [ ]:
# overall detection summary
detection_summary = overall_detection_table(df)
print(detection_summary)

In [ ]:
# directional detection summary
directional_detection = direction_detection_table(df)
print(directional_detection)

In [ ]:
images = detect_minutiae(image_folder, max_block=96, minutia_quality_threshold=0.6)

In [ ]:
minutiae_type = 'minutiae'
plot_minutiae_map(images[0], source=minutiae_type, marker_size=70, saveto=None)#f'{save_folder}/{prefix}_minutiae_{minutiae_type}_region.png')
plot_fusion_set_size(df, source=minutiae_type, saveto=None)#f'{save_folder}/{prefix}_{minutiae_type}_direction_fusion.png')

In [ ]:
# One impression — manually measured distal-axis angle
plot_minutiae_by_illumination(images, distal_angle=-1.5848,
                                  saveto=None)#f'{save_folder}/{prefix}_ridge_flow_illumination_alignment_stacked.png')

In [ ]:
def illumination_angles(distal_angle):
    return {"North": distal_angle,
            "East": distal_angle + np.pi / 2,
            "South": distal_angle + np.pi,
            "West": distal_angle - np.pi / 2,}

def axial_alignment(ridge_angle, light_angle):
    """Both inputs in radians; output in degrees (0–90)."""
    difference = (ridge_angle - light_angle) % np.pi
    return np.degrees(np.minimum(difference, np.pi - difference))

def evaluate_alignment_distributions(images, distal_axis_angle):
    distal_axis_angle = (distal_axis_angle - np.pi / 2) % (2 * np.pi)
    directions = illumination_angles(distal_axis_angle)
    bins = np.arange(0, 91, 15)

    histogram_rows = []
    summary_rows = []

    for image in images:
        direction = image.direction.capitalize()
        light_angle = directions[direction]

        theta = np.asarray(image.theta)
        mask = np.asarray(image.normalisedROIMask, dtype=bool)

        if theta.shape != mask.shape:
            raise ValueError(f'{direction}: theta and mask shapes differ')

        valid = mask & np.isfinite(theta)

        if not np.any(valid):
            continue

        # Pixel-wise ridge–illumination alignment
        alignment_map = axial_alignment(theta, light_angle)
        area_angles = alignment_map[valid]

        area_counts, _ = np.histogram(area_angles, bins=bins)
        area_pct = 100 * area_counts / area_counts.sum()

        # Minutiae coordinates are already scaled full-image coordinates
        y0, x0 = np.asarray(image.roiCoords) * image.median_scale

        minutiae_angles = []

        for minutia in image.minutiae:
            x = int(round(minutia.x - x0))
            y = int(round(minutia.y - y0))

            if ( 0 <= y < theta.shape[0] and 0 <= x < theta.shape[1] and valid[y, x]):
                minutiae_angles.append(alignment_map[y, x])

        minutiae_counts, _ = np.histogram(minutiae_angles, bins=bins)

        if minutiae_counts.sum() > 0:
            minutiae_pct = (100 * minutiae_counts / minutiae_counts.sum())
        else:
            minutiae_pct = np.zeros(len(bins) - 1)

        for i in range(len(bins) - 1):
            histogram_rows.append({'Direction': direction, 'AngleMin': bins[i], 'AngleMax': bins[i + 1],
                                   'AreaPercentage': area_pct[i], 'MinutiaePercentage': minutiae_pct[i],})

        summary_rows.append({'Direction': direction, 'Area45to90': float(100 * np.mean(area_angles >= 45)),
                             'Minutiae45to90': float(100 * np.mean(np.asarray(minutiae_angles) >= 45)) if minutiae_angles else np.nan,
                             'MinutiaeTotal': len(image.minutiae), 'MinutiaeInMask': len(minutiae_angles)})

    return pd.DataFrame(histogram_rows), pd.DataFrame(summary_rows)

In [ ]:
alignment_cmap = LinearSegmentedColormap.from_list('alignmen', ['#0000FF', '#FF0000'])

# def plot_alignment_histograms(hist_df):
#     directions = ["North", "East", "South", "West"]

#     fig, axes = plt.subplots(1, 4, figsize=(11, 2.8), sharex=True, sharey=True,
#                              constrained_layout=True)

#     for ax, direction in zip(axes, directions):
#         data = hist_df[hist_df["Direction"] == direction]

#         ax.axvspan(45, 90, color="0.92", zorder=0)
#         ax.bar(data["AngleMin"] + 7.5, data["Percentage"], width=13, color="0.35", zorder=2)

#         perpendicular_area = data.loc[data["AngleMin"] >= 45, "Percentage"].sum()

#         ax.set_title(f"{direction}\n{perpendicular_area:.1f}% near-perpendicular")
#         ax.set_xlim(0, 90)
#         ax.set_xticks([0, 30, 60, 90])
#         ax.spines[["top", "right"]].set_visible(False)

#     axes[0].set_ylabel("Valid impression area (%)")
#     fig.supxlabel("Ridge–illumination angle (°)")

#     plt.show()

def plot_colored_ridge_flow(ax, theta, valid, light_angle, spacing=16, alpha=0.55):
    h, w = theta.shape

    # Same sampling positions and line length as pyfing
    yy, xx = np.mgrid[0:h:spacing, 0:w:spacing]
    keep = valid[yy, xx]

    x = xx[keep]
    y = yy[keep]
    angles = theta[y, x]

    # pyfing's half-length formula, with scale=1
    hl = max(1, (spacing - 1) / 2)

    dx = np.round(np.cos(angles) * hl)
    dy = np.round(-np.sin(angles) * hl)

    segments = np.stack([np.stack([x - dx, y - dy], axis=-1), np.stack([x + dx, y + dy], axis=-1)], axis=1)

    lines = LineCollection(segments, cmap=alignment_cmap, norm=plt.Normalize(0, 90), linewidths=1.5, alpha=alpha)

    lines.set_array(axial_alignment(angles, light_angle))

    ax.add_collection(lines)
    ax.set_xlim(-0.5, w - 0.5)
    ax.set_ylim(h - 0.5, -0.5)
    ax.set_aspect('equal')

    return lines

def plot_alignment_figure(images, distal_axis_angle, hist_df, summary_df, save_to=None):
    distal_axis_angle = (distal_axis_angle - np.pi / 2) % (2 * np.pi)
    directions = illumination_angles(distal_axis_angle)
    order = ['North', 'East', 'South', 'West']

    image_by_direction = {image.direction.capitalize(): image for image in images}

    fig, axes = plt.subplots(2, 4, figsize=(13, 7), gridspec_kw={'height_ratios': [1.8, 1]}, 
                             constrained_layout=True, dpi=300)

    alignment_artist = None

    for col, direction in enumerate(order):
        image = image_by_direction[direction]

        theta = np.asarray(image.theta)
        mask = np.asarray(image.normalisedROIMask, dtype=bool)
        y0, x0 = np.asarray(image.roiCoords) * image.median_scale
        minutiae = image.minutiae
        updated_minutiae = [m._replace(x=int(m.x - x0), y=int(m.y - y0)) for m in minutiae]
        minutiae = updated_minutiae
        valid = mask & np.isfinite(theta)
        img = image.normalisedROI

        ax_img = axes[0, col]
        ax_img.imshow(cv2.cvtColor(image.normalisedROIC, cv2.COLOR_BGR2RGB), origin='upper', interpolation='nearest')
        
        alignment_artist = plot_colored_ridge_flow(ax_img, theta, valid, directions[direction], spacing=32, alpha=0.90)

        # Convert full-image minutiae positions to ROI positions
        y0, x0 = np.asarray(image.roiCoords) * image.median_scale
        xs, ys = [], []
        for minutia in image.minutiae:
            x = int(round(minutia.x - x0))
            y = int(round(minutia.y - y0))

            if (0 <= y < theta.shape[0] and 0 <= x < theta.shape[1] and valid[y, x]):
                xs.append(x)
                ys.append(y)

        ax_img.scatter(xs, ys, s=20, c='#DFFF00', edgecolors='black', linewidths=0.8, zorder=3)

        ax_img.set_title(direction, fontsize=11)
        ax_img.axis('off')

        # Histogram
        ax_hist = axes[1, col]
        data = hist_df[hist_df['Direction'] == direction]
        summary = summary_df[summary_df['Direction'] == direction].iloc[0]

        centers = data['AngleMin'].to_numpy() + 7.5

        ax_hist.axvspan(45, 90, color='0.94', zorder=0)
        ax_hist.bar(centers - 3.5, data['AreaPercentage'], width=7, color='0.7', label='Impression area', zorder=2)
        ax_hist.bar(centers + 3.5, data['MinutiaePercentage'], width=7, color='#4477AA', label='Detected minutiae', zorder=2)
        ax_hist.set_title(f"45–90°: {summary['Area45to90']:.0f}% area, {summary['Minutiae45to90']:.0f}% minutiae", fontsize=9)
        ax_hist.set_xlim(0, 90)
        ax_hist.set_xticks([0, 30, 60, 90])
        ax_hist.set_xlabel('Alignment angle (°)', fontsize=9)
        ax_hist.spines[['top', 'right']].set_visible(False)

        if col == 0:
            ax_hist.set_ylabel("Percentage (%)")

    # Shared colorbar
    fig.colorbar(alignment_artist, ax=axes[0, :], label='Ridge–illumination angle (°)', shrink=0.75, pad=0.02)
    handles, labels = axes[1, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='outside lower center', ncol=2, frameon=False)
    if save_to is not None:
        plt.savefig(save_to)
    plt.show()

In [ ]:
hist_df, summary_df = evaluate_alignment_distributions(images[0], distal_axis_angle=2.213)

plot_alignment_figure(images[0], distal_axis_angle=2.213, hist_df=hist_df, summary_df=summary_df,
                      save_to=None)#f'{save_folder}/{prefix}_ridge_flow_illumination_alignment_individual.png')

print(summary_df)